# Modélisation Topographique et Hydrologique : Département de la Gironde

Ce projet contient une chaîne de traitement automatisée en Python visant à extraire un triptyque de variables topographiques à partir d'un Modèle Numérique de Terrain (MNT) à haute résolution (5 mètres). 

L'objectif final est la production et la consolidation de trois couches de données fondamentales en géomorphologie et en viticulture :
1. **La Pente (Slope)**
2. **L'Exposition (Aspect)**
3. **L'Indice d'Humidité Topographique (TWI)**

## Architecture du Pipeline de Traitement

Le traitement est divisé en 8 phases séquentielles.

### Phase 1 : Fusion et Nettoyage (MNT de base)
* **Fichier produit :** `01_mnt_altitude_ign.tif`
* **Fonction :** Assemblage des dalles brutes issues de l'IGN et correction des valeurs aberrantes.
* **Algorithme :** Le script lit les matrices de pixels individuelles, les aligne sur une grille commune, et applique un masque matriciel conditionnel : toute valeur strictement égale à -99999 est reclassifiée avec le marqueur système `NoData`.
* **Justification :** Évite que les algorithmes de voisinage n'interprètent les bordures du département (codées à -99999.0 par l'IGN) comme des variations de relief réelles.

### Phase 2 : Calcul de la Pente (Slope)
* **Fichier produit :** `02_pente_degres.tif`
* **Fonction :** Détermination de l'inclinaison du terrain.
* **Algorithme :** Utilise une fenêtre d'analyse mobile (5x5 pixels, méthode polynomiale de Taylor bivariée du 3e ordre). L'algorithme calcule le taux de changement maximum de l'élévation entre la cellule centrale et ses voisines directes.
* **Justification :** Doit être calculée sur le MNT non altéré (avant remplissage des dépressions) pour conserver la mesure physique exacte du terrain (talus, fossés).

### Phase 3 : Calcul de l'Exposition (Aspect)
* **Fichier produit :** `03_aspect_degres.tif`
* **Fonction :** Détermination de l'orientation des versants par rapport au Nord géographique.
* **Algorithme :** Sur la même fenêtre 5x5, l'algorithme détermine la direction (l'angle azimutal) vers laquelle pointe le vecteur de la pente maximale calculé précédemment. Les surfaces sans vecteur de pente (pente = 0) reçoivent la valeur -1.
* **Justification :** Fournit la base de l'analyse microclimatique d'une parcelle.

### Phase 4 : Remplissage des Dépressions (Fill Depressions)
* **Fichier produit :** `04_mnt_rempli.tif` (Fichier de transition)
* **Fonction :** Création d'un MNT "hydrologiquement correct".
* **Algorithme :** Parcours itératif de la matrice pour identifier les pixels "puits" (n'ayant aucun voisin d'altitude inférieure). L'algorithme incrémente virtuellement l'altitude de ces pixels jusqu'à ce qu'un chemin d'écoulement descendant continu soit trouvé vers les bords de la carte.
* **Justification :** Étape préparatoire obligatoire pour le routage de l'eau. Sans cela, les calculs de flux s'arrêteraient dans chaque irrégularité locale du terrain.

### Phase 5 : Accumulation de Flux (Flow Accumulation)
* **Fichier produit :** `05_accumulation_flux.tif` (Fichier de transition)
* **Fonction :** Calcul de la Surface de Drainage Spécifique (SCA).
* **Algorithme :** Méthode D-Infinity (Tarboton, 1997). Au lieu de forcer l'eau à s'écouler vers un seul des 8 pixels voisins (méthode D8), l'algorithme modélise le flux comme un vecteur continu glissant sur des facettes, distribuant l'accumulation de manière proportionnelle sur plusieurs pixels en aval.
* **Justification :** Fournit une modélisation mathématique plus réaliste et lisse des écoulements, particulièrement sur les versants peu inclinés.

### Phase 6 : Indice d'Humidité Topographique (TWI)
* **Fichier produit :** `06_twi_indice_humidite.tif`
* **Fonction :** Synthèse de la dynamique de l'eau.
* **Algorithme :** Applique l'équation `TWI = ln(a / tan(β))` pour chaque pixel, où `a` est la surface de drainage spécifique (issue de la Phase 5) et `β` est la pente locale en radians (issue de la Phase 2). Une routine gère les divisions par zéro potentielles sur les zones de pente nulle.
* **Justification :** Standardise des valeurs d'accumulation extrêmes en un indice continu quantifiant la propension d'un pixel à accumuler l'eau par rapport à sa capacité à la drainer.

### Phase 7 : Consolidation (Stacking)
* **Fichier produit :** `GIRONDE_TOPO_STACK_5M.tif`
* **Fonction :** Création d'un fichier maître.
* **Algorithme :** Concaténation de matrices bidimensionnelles distinctes (Altitude, Pente, Aspect, TWI) pour former un tableau tridimensionnel unique (GeoTIFF multi-bandes), synchronisant les coordonnées spatiales de toutes les variables.
* **Justification :** Rationalisation du stockage et préparation des données pour les outils de Machine Learning ou d'analyse spatiale multicritères.

### Phase 8 : Tableau de Bord Géostatistique
* **Fichier produit :** `07_analyse_topo_scientifique.png`
* **Fonction :** Analyse exploratoire et vérification de la cohérence des distributions.
* **Algorithme :** Sous-échantillonnage de la matrice (réduction d'échelle spatiale) suivi du calcul de quantiles, de la détermination de la moyenne circulaire (via décomposition de vecteurs sinus/cosinus pour l'Aspect), et de l'ajustement exponentiel des classes (log-bins) pour l'affichage des variables à forte asymétrie.
* **Justification :** Permet la validation technique du modèle final sans saturer la mémoire vive du système d'analyse.

In [4]:
!pip install whitebox rasterio

In [2]:
import os
import glob
import rasterio
import numpy as np
from rasterio.merge import merge

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
# Dossier contenant les 500 fichiers .asc de l'IGN
input_folder = r"C:\Users\tliegeon\Desktop\RGEALTI_MNT_5"
# Dossier de sortie pour le MNT fusionné
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
os.makedirs(output_dir, exist_ok=True)

# Paramètres techniques
mnt_final_tif = os.path.join(output_dir, "01_mnt_altitude_ign.tif")
NODATA_VAL = -99999.0
DTYPE = 'float32'
EPSG_LAMBERT_93 = "EPSG:2154"

# ---------------------------------------------------------
# 2. FUSION ET INJECTION DU CRS
# ---------------------------------------------------------
try:
    print("Étape : Recherche et ouverture des dalles .asc...")
    all_asc = sorted(glob.glob(os.path.join(input_folder, "*.asc")))
    
    if not all_asc:
        raise FileNotFoundError(f"Aucun fichier .asc trouvé dans : {input_folder}")

    # Ouverture des sources
    srcs = [rasterio.open(f) for f in all_asc]
    
    print(f"Fusion de {len(all_asc)} dalles en cours (cela peut prendre quelques minutes)...")
    # Fusion géométrique
    mosaic, transform = merge(srcs)
    
    # Récupération des métadonnées de base depuis la première dalle
    meta = srcs[0].meta.copy()
    
    # Fermeture des fichiers sources pour libérer la mémoire vive
    for s in srcs:
        s.close()

    # Mise à jour des métadonnées pour le fichier final
    meta.update(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        crs=EPSG_LAMBERT_93,  # Injection indispensable pour WhiteboxTools
        count=1,
        dtype=DTYPE,
        nodata=NODATA_VAL,
        compress='DEFLATE',   # Compression pour réduire la taille sur disque
        tiled=True            # Optimisation pour la lecture par blocs
    )

    print("Écriture du fichier MNT fusionné...")
    with rasterio.open(mnt_final_tif, 'w', **meta) as dst:
        # Conversion en float32 et gestion des NaN
        data_clean = mosaic[0].astype(DTYPE)
        data_clean = np.where(np.isnan(data_clean), NODATA_VAL, data_clean)
        dst.write(data_clean, 1)

    print(f"\nSuccès : Le MNT fusionné est disponible ici : {mnt_final_tif}")

except Exception as e:
    print(f"\nErreur lors de la fusion : {e}")

# ---------------------------------------------------------
# 3. VÉRIFICATION DU PRODUIT
# ---------------------------------------------------------
if os.path.exists(mnt_final_tif):
    with rasterio.open(mnt_final_tif) as check:
        print("\n--- Diagnostic du fichier produit ---")
        print(f"Dimensions : {check.width} x {check.height} pixels")
        print(f"Système de coordonnées (CRS) : {check.crs}")
        print(f"Résolution spatiale : {check.res}")
        print(f"Valeur NoData : {check.nodata}")

Étape : Recherche et ouverture des dalles .asc...
Fusion de 497 dalles en cours (cela peut prendre quelques minutes)...
Écriture du fichier MNT fusionné...

Succès : Le MNT fusionné est disponible ici : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\01_mnt_altitude_ign.tif

--- Diagnostic du fichier produit ---
Dimensions : 26000 x 33000 pixels
Système de coordonnées (CRS) : EPSG:2154
Résolution spatiale : (5.0, 5.0)
Valeur NoData : -99999.0


In [3]:
import os
import rasterio
import numpy as np
from whitebox.whitebox_tools import WhiteboxTools

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
mnt_source = os.path.join(output_dir, "01_mnt_altitude_ign.tif")
pente_out  = os.path.join(output_dir, "02_pente_degres.tif")

wbt = WhiteboxTools()
wbt.verbose = False

NODATA_VAL = -99999.0
DTYPE = 'float32'

# ---------------------------------------------------------
# 2. CALCUL DE LA PENTE
# ---------------------------------------------------------
try:
    print("Calcul de la pente en cours...")
    
    # L'unité 'degrees' est impérative pour le calcul du TWI
    wbt.slope(
        dem=mnt_source, 
        output=pente_out, 
        units="degrees"
    )

    # ---------------------------------------------------------
    # 3. NORMALISATION DU FICHIER (NODATA & DTYPE)
    # ---------------------------------------------------------
    print("Normalisation du fichier de pente...")
    tmp_path = pente_out + ".tmp.tif"
    
    with rasterio.open(pente_out) as src:
        meta = src.meta.copy()
        meta.update(dtype=DTYPE, nodata=NODATA_VAL)
        data = src.read(1).astype(DTYPE)
        
        # Sécurité : remplacement des NaN ou valeurs aberrantes par NoData
        data = np.where((np.isnan(data)) | (data < 0) | (data > 90), NODATA_VAL, data)
        
    with rasterio.open(tmp_path, 'w', **meta) as dst:
        dst.write(data, 1)
    
    os.replace(tmp_path, pente_out)
    print(f"Succès : Fichier de pente généré : {pente_out}")

except Exception as e:
    print(f"Erreur lors du calcul de la pente : {e}")

# ---------------------------------------------------------
# 4. VÉRIFICATION DES VALEURS
# ---------------------------------------------------------
if os.path.exists(pente_out):
    with rasterio.open(pente_out) as src:
        raster_data = src.read(1)
        valid_values = raster_data[raster_data != NODATA_VAL]
        
        print("\n--- Diagnostic de la couche Pente ---")
        print(f"Pente minimale : {np.min(valid_values):.2f}°")
        print(f"Pente maximale : {np.max(valid_values):.2f}°")
        print(f"Moyenne du département : {np.mean(valid_values):.2f}°")

Calcul de la pente en cours...
Normalisation du fichier de pente...
Succès : Fichier de pente généré : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\02_pente_degres.tif

--- Diagnostic de la couche Pente ---
Pente minimale : 0.00°
Pente maximale : 76.23°
Moyenne du département : 2.32°


In [4]:
import os
import rasterio
import numpy as np
from whitebox.whitebox_tools import WhiteboxTools

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
mnt_source = os.path.join(output_dir, "01_mnt_altitude_ign.tif")
aspect_out = os.path.join(output_dir, "03_aspect_degres.tif")

wbt = WhiteboxTools()
wbt.verbose = False

NODATA_VAL = -99999.0
DTYPE = 'float32'

# ---------------------------------------------------------
# 2. CALCUL DE L'EXPOSITION
# ---------------------------------------------------------
try:
    print("Calcul de l'exposition (Aspect) en cours...")
    
    wbt.aspect(
        dem=mnt_source, 
        output=aspect_out
    )

    # ---------------------------------------------------------
    # 3. NORMALISATION DU FICHIER
    # ---------------------------------------------------------
    print("Normalisation du fichier d'exposition...")
    tmp_path = aspect_out + ".tmp.tif"
    
    with rasterio.open(aspect_out) as src:
        meta = src.meta.copy()
        meta.update(dtype=DTYPE, nodata=NODATA_VAL)
        data = src.read(1).astype(DTYPE)
        
        # WhiteboxTools utilise souvent -1.0 pour les zones plates (pas d'orientation)
        # On uniformise les NaN et les valeurs hors limites vers le NoData
        data = np.where((np.isnan(data)) | (data < -1) | (data > 360), NODATA_VAL, data)
        
    with rasterio.open(tmp_path, 'w', **meta) as dst:
        dst.write(data, 1)
    
    os.replace(tmp_path, aspect_out)
    print(f"Succès : Fichier d'exposition généré : {aspect_out}")

except Exception as e:
    print(f"Erreur lors du calcul de l'exposition : {e}")

# ---------------------------------------------------------
# 4. DIAGNOSTIC
# ---------------------------------------------------------
if os.path.exists(aspect_out):
    with rasterio.open(aspect_out) as src:
        raster_data = src.read(1)
        valid_values = raster_data[raster_data != NODATA_VAL]
        
        print("\n--- Diagnostic de la couche Aspect ---")
        print(f"Orientation min : {np.min(valid_values):.2f}°")
        print(f"Orientation max : {np.max(valid_values):.2f}°")
        print("Note : 0=Nord, 90=Est, 180=Sud, 270=Ouest, -1=Plat")

Calcul de l'exposition (Aspect) en cours...
Normalisation du fichier d'exposition...
Succès : Fichier d'exposition généré : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\03_aspect_degres.tif

--- Diagnostic de la couche Aspect ---
Orientation min : -1.00°
Orientation max : 360.00°
Note : 0=Nord, 90=Est, 180=Sud, 270=Ouest, -1=Plat


In [5]:
import os
import rasterio
import numpy as np
from whitebox.whitebox_tools import WhiteboxTools

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
mnt_source = os.path.join(output_dir, "01_mnt_altitude_ign.tif")
mnt_filled = os.path.join(output_dir, "04_mnt_rempli.tif")

wbt = WhiteboxTools()
wbt.verbose = False

NODATA_VAL = -99999.0
DTYPE = 'float32'

# ---------------------------------------------------------
# 2. CALCUL : REMPLISSAGE DES CUVETTES
# ---------------------------------------------------------
try:
    print("Correction du MNT (Remplissage des dépressions)...")
    
    wbt.fill_depressions(
        dem=mnt_source, 
        output=mnt_filled
    )

    # ---------------------------------------------------------
    # 3. NORMALISATION (NODATA & DTYPE)
    # ---------------------------------------------------------
    print("Normalisation du fichier...")
    tmp_path = mnt_filled + ".tmp.tif"
    
    with rasterio.open(mnt_filled) as src:
        meta = src.meta.copy()
        meta.update(dtype=DTYPE, nodata=NODATA_VAL)
        data = src.read(1).astype(DTYPE)
        data = np.where(np.isnan(data), NODATA_VAL, data)
        
    with rasterio.open(tmp_path, 'w', **meta) as dst:
        dst.write(data, 1)
    
    os.replace(tmp_path, mnt_filled)
    print(f"Succès : MNT hydrologique généré : {mnt_filled}")

except Exception as e:
    print(f"Erreur : {e}")

# ---------------------------------------------------------
# 4. DIAGNOSTIC
# ---------------------------------------------------------
if os.path.exists(mnt_filled):
    with rasterio.open(mnt_filled) as src:
        print("\n--- Diagnostic MNT Rempli ---")
        print(f"Altitude min : {np.nanmin(src.read(1)[src.read(1) != NODATA_VAL]):.2f} m")
        print("Note : Ce fichier ressemblera visuellement au MNT original, c'est normal.")

Correction du MNT (Remplissage des dépressions)...
Normalisation du fichier...
Succès : MNT hydrologique généré : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\04_mnt_rempli.tif

--- Diagnostic MNT Rempli ---
Altitude min : -2.17 m
Note : Ce fichier ressemblera visuellement au MNT original, c'est normal.


In [6]:
import os
import rasterio
import numpy as np
from whitebox.whitebox_tools import WhiteboxTools

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
mnt_filled = os.path.join(output_dir, "04_mnt_rempli.tif")
flow_acc   = os.path.join(output_dir, "05_accumulation_flux.tif")

wbt = WhiteboxTools()
wbt.verbose = False

NODATA_VAL = -99999.0
DTYPE = 'float32'

# ---------------------------------------------------------
# 2. CALCUL : ACCUMULATION DE FLUX
# ---------------------------------------------------------
try:
    print("Calcul de l'accumulation de flux (D-Infinity) avec paramètre 'i'...")
    
    # Utilisation de 'i' (shorthand pour input) qui est le plus stable dans WBT
    wbt.d_inf_flow_accumulation(
        i=mnt_filled, 
        output=flow_acc,
        out_type="sca"
    )

    # ---------------------------------------------------------
    # 3. NORMALISATION
    # ---------------------------------------------------------
    if os.path.exists(flow_acc):
        print("Normalisation du fichier...")
        tmp_path = flow_acc + ".tmp.tif"
        
        with rasterio.open(flow_acc) as src:
            meta = src.meta.copy()
            meta.update(dtype=DTYPE, nodata=NODATA_VAL)
            data = src.read(1).astype(DTYPE)
            data = np.where(np.isnan(data), NODATA_VAL, data)
            
        with rasterio.open(tmp_path, 'w', **meta) as dst:
            dst.write(data, 1)
        
        os.replace(tmp_path, flow_acc)
        print(f"Succès : Accumulation générée : {flow_acc}")
    else:
        print("Erreur : Le fichier de sortie n'a pas été créé par WhiteboxTools.")

except Exception as e:
    print(f"Erreur technique rencontrée : {e}")

# ---------------------------------------------------------
# 4. DIAGNOSTIC
# ---------------------------------------------------------
if os.path.exists(flow_acc):
    with rasterio.open(flow_acc) as src:
        valid_values = src.read(1)[src.read(1) != NODATA_VAL]
        print("\n--- Diagnostic Flux ---")
        print(f"SCA min : {np.min(valid_values):.2f} m²")
        print(f"SCA max : {np.max(valid_values):.2f} m²")

Calcul de l'accumulation de flux (D-Infinity) avec paramètre 'i'...
Normalisation du fichier...
Succès : Accumulation générée : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\05_accumulation_flux.tif

--- Diagnostic Flux ---
SCA min : 5.00 m²
SCA max : 1507331712.00 m²


In [1]:
import os
import rasterio
import numpy as np
from whitebox.whitebox_tools import WhiteboxTools

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"
pente_tif  = os.path.join(output_dir, "02_pente_degres.tif")
flow_acc   = os.path.join(output_dir, "05_accumulation_flux.tif")
twi_tif    = os.path.join(output_dir, "06_twi_indice_humidite.tif")

wbt = WhiteboxTools()
wbt.verbose = False

NODATA_VAL = -99999.0
DTYPE = 'float32'

# ---------------------------------------------------------
# 2. CALCUL : INDICE D'HUMIDITÉ TOPOGRAPHIQUE
# ---------------------------------------------------------
try:
    print("Calcul de l'indice TWI en cours...")
    
    wbt.wetness_index(
        sca=flow_acc, 
        slope=pente_tif, 
        output=twi_tif
    )

    # ---------------------------------------------------------
    # 3. NORMALISATION DU FICHIER
    # ---------------------------------------------------------
    print("Normalisation du TWI...")
    tmp_path = twi_tif + ".tmp.tif"
    
    with rasterio.open(twi_tif) as src:
        meta = src.meta.copy()
        meta.update(dtype=DTYPE, nodata=NODATA_VAL)
        data = src.read(1).astype(DTYPE)
        
        # Le calcul du TWI peut générer des valeurs infinies (Inf) si la pente est de 0
        # Nous les neutralisons en les passant en NoData pour sécuriser les futurs modèles
        data = np.where(np.isnan(data) | np.isinf(data), NODATA_VAL, data)
        
    with rasterio.open(tmp_path, 'w', **meta) as dst:
        dst.write(data, 1)
    
    os.replace(tmp_path, twi_tif)
    print(f"Succès : Indice TWI généré : {twi_tif}")

except Exception as e:
    print(f"Erreur lors du calcul du TWI : {e}")

# ---------------------------------------------------------
# 4. DIAGNOSTIC DU TWI
# ---------------------------------------------------------
if os.path.exists(twi_tif):
    with rasterio.open(twi_tif) as src:
        data = src.read(1)
        valid = data[data != NODATA_VAL]
        
        print("\n--- Diagnostic TWI Final ---")
        print(f"TWI min : {np.min(valid):.2f}")
        print(f"TWI max : {np.max(valid):.2f}")
        print(f"Moyenne : {np.mean(valid):.2f}")

Calcul de l'indice TWI en cours...
Normalisation du TWI...
Succès : Indice TWI généré : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\06_twi_indice_humidite.tif

--- Diagnostic TWI Final ---
TWI min : 0.38
TWI max : 33.49
Moyenne : 7.00


In [2]:
import os
import rasterio

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
output_dir = r"C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde"

# Fichiers à conserver et à empiler
couches_utiles = {
    1: ("Altitude_m", os.path.join(output_dir, "01_mnt_altitude_ign.tif")),
    2: ("Pente_deg", os.path.join(output_dir, "02_pente_degres.tif")),
    3: ("Aspect_deg", os.path.join(output_dir, "03_aspect_degres.tif")),
    4: ("TWI_Indice", os.path.join(output_dir, "06_twi_indice_humidite.tif"))
}

fichier_stack_final = os.path.join(output_dir, "GIRONDE_TOPO_STACK_5M.tif")

# ---------------------------------------------------------
# 2. CRÉATION DU FICHIER MULTI-BANDES (STACK)
# ---------------------------------------------------------
try:
    print("Création du fichier consolidé (Stack) en cours...")
    
    # Lecture des métadonnées du fichier source (MNT) pour définir le format
    with rasterio.open(couches_utiles[1][1]) as src0:
        meta = src0.meta.copy()
        
    # Mise à jour des métadonnées pour 4 bandes
    meta.update(count=len(couches_utiles))
    
    # Création et écriture du nouveau fichier
    with rasterio.open(fichier_stack_final, 'w', **meta) as dst:
        for index_bande, (nom_bande, chemin_fichier) in couches_utiles.items():
            if os.path.exists(chemin_fichier):
                print(f"Intégration de la bande {index_bande} : {nom_bande}...")
                with rasterio.open(chemin_fichier) as src:
                    dst.write(src.read(1), index_bande)
                    dst.set_band_description(index_bande, nom_bande)
            else:
                print(f"Erreur : Le fichier {chemin_fichier} est introuvable.")

    print(f"\nSuccès : Le fichier final est disponible ici : {fichier_stack_final}")
    print("Information : Aucune donnée intermédiaire n'a été supprimée.")

except Exception as e:
    print(f"Erreur lors de la création du stack : {e}")

Création du fichier consolidé (Stack) en cours...
Intégration de la bande 1 : Altitude_m...
Intégration de la bande 2 : Pente_deg...
Intégration de la bande 3 : Aspect_deg...
Intégration de la bande 4 : TWI_Indice...

Succès : Le fichier final est disponible ici : C:\Users\tliegeon\Desktop\indice\sortie\topographie_gironde\GIRONDE_TOPO_STACK_5M.tif
Information : Aucune donnée intermédiaire n'a été supprimée.
